# 写真套图双模式运行示例 (Photobook Suite: Dual-Mode Guide)

本 Notebook 演示如何使用 **通用写真套图引擎（`scripts/run_photobook.py`）** 实现：
- **方式 1：T2I 文本高密度生图**（速度极快、光影氛围与皮肤质感极致，适合常规写意姿态）
- **方式 2：VNCCS 3D 骨骼控制生图**（100% 锁定 3D 关节姿态，适合极限拉伸、瑜伽、下腰等高难度动作）

---

In [ ]:
# ===== 1. 环境准备与模块导入 =====
import os
import sys
from pathlib import Path

# 确保项目根目录在 Python 检索路径中
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from scripts.run_photobook import list_available_templates, execute_photobook
print("✅ 成功导入写真引擎核心方法 execute_photobook！")

In [ ]:
# ===== 2. 查看当前所有可用写真模板及其控姿模式 =====
templates = list_available_templates()
print(f"发现 {len(templates)} 套写真模板：\n")
for tid, tpath in templates.items():
    import yaml
    data = yaml.safe_load(tpath.read_text(encoding='utf-8'))
    name_zh = data.get('name_zh', tid)
    shots = data.get('shots', [])
    has_pose = any('pose' in s for s in shots)
    tag = "🎮 [支持方式2: 3D骨骼绑定]" if has_pose else "✍️ [方式1: 纯T2I文本]"
    print(f"  • {tid:22s} | {name_zh} ({len(shots)} 幕分镜) {tag}")

## 方式 1 演示：T2I 纯文本高密度生图

以新角色 **[CN-19] 苏青荷** 套用古典闺房模板（`chinese_canopy_bed`）为例：
- 提示词承担所有任务（人物特征 + 肢体自然语言描述 + 场景光影）；
- 无需 3D 浏览器，单图约 30~45 秒。

In [ ]:
# ===== 3. 运行方式 1：苏青荷 · 古典闺房写真 (Dry-run 预览) =====
res_mode1 = execute_photobook(
    character="cn19_su_qinghe",           # 新角色：苏青荷
    template="chinese_canopy_bed",        # 古典拔步床模板
    mode="t2i",                           # 显式指定方式 1 (T2I)
    dry_run=True,                         # 先预览提示词
    run=False                             # 设为 True 则提交 ComfyUI 真正出图
)

## 方式 2 演示：VNCCS 3D 骨骼精准姿态控制

以新角色 **[EA-05] 韩敏智** 套用 3D 骨骼写真模板（`pose3d_studio`）为例：
- 模板中每一幕指定了 `pose: "Kneeling Leaning Back"`、`"Squatting Legs Spread"` 等 3D 骨骼文件名；
- 系统会自动调用 VNCCS 3D 傀儡假人固定关节角度，彻底防止高难度动作四肢畸变。

In [ ]:
# ===== 4. 运行方式 2：韩敏智 · 3D 骨骼精准控制写真 =====
res_mode2 = execute_photobook(
    character="ea05_han_minji",           # 新角色：韩敏智 (普拉提健美超模)
    template="pose3d_studio",             # 3D 姿势绑定模板
    mode="auto",                          # auto 模式下检测到 pose 字段自动走方式 2
    export_dir="photobooks/han_minji_pose3d",
    dry_run=True,                         # 预览生成的提示词与 3D 姿势分配
    run=False                             # 设为 True 则直接提交生成
)

## 深入解答：方式 2 是不是不需要用提示词 txt 了？

**答案：不是！方式 2 依然需要提示词 txt，但提示词的分工彻底变了！**

| 对比项 | 方式 1 (T2I 纯文本生图) | 方式 2 (VNCCS 3D 骨骼控姿) |
| :--- | :--- | :--- |
| **姿态四肢由谁控制** | **全部写在提示词里**（需要详尽描述手放哪里、背怎么拱、膝盖弯几度） | **由 3D 假人骨骼决定**（提示词里不再需要写繁琐的关节角度） |
| **提示词核心任务** | 描写人物 + 描写动作 + 描写场景 + 光影画质 | **专心描写：人物外观 + 场景光影 + 皮肤水光与质感** |
| **提示词文件 (.txt)** | 必须有，长达 500~1000 字符 | **必须有**，作为场景与主体描述注入到 3D 渲染图上 |

> **核心原理**：3D 木偶只是一具“无脸白模假人”。AI 需要根据你提供的提示词，将这个 3D 假人“穿上衣服或裸体皮肤”、“画上苏青荷或韩敏智的面容”，并“渲染上床榻、烛光或落地窗光影”。所以提示词不可或缺！

In [ ]:
# ===== 5. 查看导出的方式 2 提示词与参数结构 =====
import json

param_path = Path("photobooks/han_minji_pose3d/01_kneeling_leaning_back.params.json")
txt_path = Path("photobooks/han_minji_pose3d/01_kneeling_leaning_back.txt")

if param_path.exists() and txt_path.exists():
    print("📄 [参数配置 .params.json]")
    print(json.dumps(json.loads(param_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
    print("\n📝 [提示词文本 .txt 内容前 200 字]")
    print(txt_path.read_text(encoding='utf-8')[:200] + "...")
else:
    print("提示：先运行上面的第 4 步导出配置")